# Card sort demo

End-to-end walkthrough of the basic card-sort analysis pipeline against the bundled test fixture: load a co-occurrence count matrix, normalize it into a similarity matrix, and visualize it as both a heatmap and a hierarchical-clustering dendrogram.

The fixture has six cards drawn from a typical e-commerce site (Shipping, Returns, Tracking, Login, Profile, Account) sorted by 20 hypothetical participants. Two clusters should be visible by inspection: a *logistics* cluster and an *account* cluster.

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

from respondent_similarity import directories
from respondent_similarity.loaders import load_cooccurrence_csv
from respondent_similarity.similarity import normalize
from respondent_similarity.plot import plot_similarity_matrix, plot_dendrogram

## 1. Load the co-occurrence count matrix

Each off-diagonal cell counts the number of participants who placed that pair of cards into the same category. Diagonal cells hold the number of participants who sorted each card.

In [ ]:
counts = load_cooccurrence_csv(directories.test_data("cooccurrence_counts.csv"))
counts

## 2. Normalize to a similarity matrix

Dividing each cell by the participant count converts the raw integer counts into proportions in `[0, 1]`. The diagonal becomes 1.0 and each off-diagonal cell is the fraction of participants who grouped that pair together.

In [ ]:
similarity = normalize(counts)
similarity

## 3. Plot the similarity matrix as a heatmap

The fixed `[0, 1]` color scale lets heatmaps from different surveys be compared directly. Two bright blocks should stand out along the diagonal — the two clusters.

### Choosing a color palette

`plot_similarity_matrix` accepts any matplotlib colormap via the `cmap` argument. A few categories are worth knowing about:

- **Sequential single-hue palettes** — `Blues`, `Greens`, `Reds`, `Purples`, `Oranges`, `Greys`. Each ramps from near-white at low values to a saturated color at high values, so cells with high similarity stand out as the most colored. Append `_r` (e.g. `Blues_r`) to reverse the direction.
- **Perceptually uniform palettes** — `viridis`, `magma`, `inferno`, `plasma`, `cividis`. These span the full lightness range and reproduce well on screen, in print, and in greyscale, but they are *not* color-to-white blends.
- **Custom blends** — when none of the built-ins match, build a `LinearSegmentedColormap` from any pair of colors.

The two cells below render the same matrix with the built-in `Blues` palette (white at low similarity, deep blue at high) and with a custom blend that runs in the opposite direction (deep blue at low similarity, white at high), so the visual effect of the palette choice is concrete.

In [ ]:
plot_similarity_matrix(similarity, title="Card sort similarity (Blues)", cmap="Blues");

In [ ]:
blue_to_white = LinearSegmentedColormap.from_list("blue_to_white", ["#0a2e5c", "white"])
plot_similarity_matrix(
    similarity,
    title="Card sort similarity (custom blue \u2192 white)",
    cmap=blue_to_white,
);

## 4. Plot a hierarchical-clustering dendrogram

Similarity is converted to distance via `1 - similarity`, then SciPy's `linkage` with average linkage is used to build the tree. The two clusters should appear as separate sub-trees that only merge at the top of the diagram.

In [ ]:
plot_dendrogram(similarity, title="Card sort dendrogram");